In [55]:
import pandas as pd
df = pd.read_csv('dataset1.csv')
df = df.drop('Unnamed: 0', axis=1)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  object 
 1   artists           113999 non-null  object 
 2   album_name        113999 non-null  object 
 3   track_name        113999 non-null  object 
 4   popularity        114000 non-null  int64  
 5   duration_ms       114000 non-null  int64  
 6   explicit          114000 non-null  bool   
 7   danceability      114000 non-null  float64
 8   energy            114000 non-null  float64
 9   key               114000 non-null  int64  
 10  loudness          114000 non-null  float64
 11  mode              114000 non-null  int64  
 12  speechiness       114000 non-null  float64
 13  acousticness      114000 non-null  float64
 14  instrumentalness  114000 non-null  float64
 15  liveness          114000 non-null  float64
 16  valence           11

In [56]:
# print(df['key'].head(20))
print(df.loc[650])

track_id            6ad7bQ48TlqAgebCExUJVe
artists                        Ichiko Aoba
album_name                               0
track_name                       いきのこり●ぼくら
popularity                              52
duration_ms                         406103
explicit                             False
danceability                         0.592
energy                              0.0833
key                                      0
loudness                           -16.517
mode                                     0
speechiness                         0.0835
acousticness                         0.946
instrumentalness                    0.0791
liveness                            0.0941
valence                              0.205
tempo                              143.243
time_signature                           4
track_genre                       acoustic
Name: 650, dtype: object


In [57]:
unique_track_genre = list(df["track_genre"].unique())
# кол-во уникальных значений равно 114
print(len(unique_track_genre))

114


In [58]:
import pandera as pa
from pandera import DataFrameSchema, Column, Check
from pandera.dtypes import Float, Int, String 

music_schema = pa.DataFrameSchema(
    columns={
        "track_id": Column(
            str,
            required=True,
            nullable=False,
            checks=[
                Check.str_length(22, 22)
            ]
        ),
        "artists": Column(
            str,
            required=True,
            nullable=False,
            checks=[
                Check.str_length(2, 512)
            ]
        ),  
        "album_name": Column(
            str,
            required=True,
            nullable=False,
            checks=[
                Check.str_length(2, 512)
            ]
        ), 
        "track_name": Column(
            str,
            required=True,
            nullable=False,
            checks=[
                Check.str_length(2, 512)
            ]
        ),          
        "popularity": Column(
            int,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 100)
            ]
        ),
        "duration_ms": Column(
            int,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 5237760)
            ]
        ),
        "explicit": Column(
            bool,
            required=True,
            nullable=False
        ),
        "danceability": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 1)
            ]
        ),
        "energy": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 1)
            ]
        ),
        "key": Column(
            int,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 11)
            ]
        ),
        "loudness": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(-45, 5)                
            ]
        ),
        "mode": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 1)
            ]
        ),
        "speechiness": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 1)
            ]
        ),
        "acousticness": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 1)
            ]
        ),
        "instrumentalness": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 1)
            ]
        ),
        "liveness": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 1)
            ]
        ),
        "valence": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 1)
            ]
        ),
        "tempo": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 256)
            ]
        ),
        "time_signature": Column(
            float,
            required=True,
            nullable=False,
            checks=[
                Check.in_range(0, 5)
            ]
        ),
        "track_genre": Column(
            str,
            required=True,
            nullable=False,
            checks=[
                 Check.isin(unique_track_genre)
            ]
        ),   
    },
    # С этой настройкой Pandera будет пытаться исправить неверные типы данных.
    coerce=True,
    # Запрещаем любые другие колонки (строгое соответствие схеме)
    strict=True,
    # Порядок колонок нам не важен.
    ordered=False
) 

In [59]:
import os
os.environ["PANDERA_VALIDATION_DEPTH"] = "SCHEMA_AND_DATA"

In [60]:
from datetime import datetime

STUDENT_NAME = "Voronova Nastya"
DATETIME = datetime.now().strftime("%Y%m%d")

try:
    music_schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as err:
    df_log = err.failure_cases
    df_log['failure_case'] = df_log['failure_case'].astype(str)
    df_log.sort_values(by='index', inplace=True)
    df_log.reset_index(drop=True, inplace=True)
    fn = f"{STUDENT_NAME}_{DATETIME}_validation_report.parquet"
    df_log.to_parquet(fn, index=False)


In [ ]:
import pandas as pd

# Проверки на корректность сформированного parquet
dfe = pd.read_parquet(fn)
print(dfe.head(50))
print(dfe.describe())

dfc = pd.read_parquet(fn, filters=[('column', '=', 'loudness')])
print(dfc.head(50))

  schema_context    column             check  check_number failure_case  \
0         Column  loudness  in_range(-45, 5)             0      -46.251   
1         Column  loudness  in_range(-45, 5)             0      -49.307   
2         Column  loudness  in_range(-45, 5)             0      -46.591   
3         Column  loudness  in_range(-45, 5)             0      -49.531   

    index  
0  101360  
1  101538  
2  101722  
3  101888  
